In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import numpy as np

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(os.path.join(path, "Q1_data.csv"))

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df['Delivery_Time'], kde=True, color='blue')
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:
print(df.isnull().sum())

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
# Task 3: Write your code here:
df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=['object']).columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()
features = df.drop('Delivery_Time', axis=1).columns
df[features] = scaler.fit_transform(df[features])

In [ ]:
# Task 6: Write your code here:
#will leave it empty

In [ ]:
# Task 1: Write your code here:
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

# Initialization for the model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf_model.fit(X_train, y_train)
    predictions = rf_model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mae_scores.append(mae)

print(f"Average MAE across folds: {np.mean(mae_scores):.4f}")

In [ ]:
# Task 1: Write your code here:
importances = rf_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10))
plt.title('Top 10 Feature Importances')
plt.show()

In [ ]:
# Task 2: Write your code here:
# Using the model from the last fold to show distribution
plt.figure(figsize=(10, 6))
sns.histplot(predictions, kde=True, color='green')
plt.title('Distribution of Predicted Delivery Times')
plt.xlabel('Predicted Delivery Time (minutes)')
plt.show()

In [ ]:
# Task Bonus: Write your code here:

#YOU SHOULD INSTALL THE LIBRARY

%pip install kagglehub catboost lightgbm tqdm -q

from catboost import CatBoostRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    #1:Random Forest
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)

    #2:CatBoost
    cb = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6, verbose=0, random_seed=42)
    cb.fit(X_train, y_train)
    cb_preds = cb.predict(X_test)

    # the avg of the prediction

    avg_preds = (rf_preds + cb_preds) / 2

    mae = mean_absolute_error(y_test, avg_preds)
    ensemble_mae_scores.append(mae)

print(f"Ensemble Average MAE: {np.mean(ensemble_mae_scores):.4f}")